In [81]:
# ============================================================
# [CELL 00] — CONFIG & IMPORTS
# Projet POKEMON — Référentiel Universel FR
# Prismora Solutions — v2.0 — 2026
# Notebook : 02_POCKET_CARDS.ipynb
# ============================================================

import requests
import pandas as pd
import json
import re
import time
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display

# Répertoire racine = dossier du notebook
ROOT = Path.cwd()
CONFIG_PATH = ROOT / "config" / "sources.json"

# Chargement config
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

SOURCE_PRIMARY   = CONFIG["sources"]["primary"]
SOURCE_SECONDARY = CONFIG["sources"]["secondary"]
SOURCE_TERTIARY  = CONFIG["sources"]["tertiary"]

# Chargement sets_master (construit par 01_SETUP_BDD)
SETS_MASTER_PATH = ROOT / CONFIG["output"]["sets_master"]
df_sets = pd.read_csv(SETS_MASTER_PATH, encoding="utf-8-sig")

# Contrôle
print("=" * 55)
print("  PROJET POKEMON — Cartes TCG Pocket FR")
print("  Prismora Solutions — 02_POCKET_CARDS.ipynb")
print("=" * 55)
print(f"\n📁 Racine       : {ROOT}")
print(f"✅ Config        : {SOURCE_PRIMARY['name']}")
print(f"✅ sets_master   : {len(df_sets)} sets chargés")
print(f"\n📋 Sets à traiter :")
display(df_sets[["code_set", "nom_fr", "nb_cartes"]].to_string(index=False))
print(f"\n📊 Total cartes attendues : {df_sets['nb_cartes'].sum()}")

  PROJET POKEMON — Cartes TCG Pocket FR
  Prismora Solutions — 02_POCKET_CARDS.ipynb

📁 Racine       : i:\Drive partagés\Prismora - Drive Principal\01_Clients\pokemon-bdd
✅ Config        : TCGdex API (FR natif)
✅ sets_master   : 19 sets chargés

📋 Sets à traiter :


"code_set                    nom_fr  nb_cartes\n      A1       Puissance Génétique        286\n PROMO-A                   Promo-A        117\n     A1a           L’Île Fabuleuse         86\n      A2      Choc Spatio-Temporel        207\n     A2a        Lumière Triomphale         96\n     A2b Réjouissances Rayonnantes        111\n      A3          Gardiens Astraux        239\n     A3a Crise Interdimensionnelle        103\n     A3b      La Clairière d'Évoli        107\n      A4 Sagesse Entre Ciel et Mer        241\n     A4a            Source Secrète        105\n     A4b        Booster de Luxe ex        379\n      B1            Méga-Ascension        331\n PROMO-B                   Promo-B         62\n     B1a      Embrasement Écarlate        103\n      B2           Parade Onirique        234\n     B2a      Merveilles de Paldea        131\n     B2b          Méga-Rayonnement        117\n      B3           Aura Palpitante        234"


📊 Total cartes attendues : 3289


In [82]:
# ============================================================
# [CELL 01] — RÉCUPÉRATION CARTES POCKET (TCGdex FR + flibustier)
# TCGdex FR en primary | flibustier en fallback si 0 cartes ou 404
# PROMO-A et PROMO-B forcés sur flibustier (TCGdex incomplet)
# ============================================================

# Sets à forcer sur flibustier
FORCER_FLIB = ["PROMO-A", "PROMO-B"]

# Tous les sets (y compris PROMO-B à 0 dans sets_master)
sets_actifs = df_sets.copy()
all_cards_index = {}
source_par_set = {}

for _, row in sets_actifs.iterrows():
    code_set    = row["code_set"]
    code_tcgdex = row["code_tcgdex"]
    nom_fr      = row["nom_fr"]

    cards = []

    # --- Forçage flibustier ou tentative TCGdex FR ---
    if code_set not in FORCER_FLIB:
        try:
            url = f"{SOURCE_PRIMARY['base_url']}/sets/{code_tcgdex}"
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            cards = resp.json().get("cards", [])
        except:
            pass

    # --- Fallback flibustier si vide, erreur ou forcé ---
    if len(cards) == 0:
        try:
            url_flib = f"{SOURCE_SECONDARY['base_url']}/cards/{code_set}.json"
            resp = requests.get(url_flib, timeout=15)
            resp.raise_for_status()
            cards = resp.json()
            source_par_set[code_set] = "flibustier"
        except Exception as e:
            source_par_set[code_set] = f"ERREUR"
            print(f"  ❌ {code_set:10s} | {nom_fr:30s} | ERREUR : {e}")
            continue
    else:
        source_par_set[code_set] = "tcgdex"

    all_cards_index[code_set] = cards
    print(f"  ✅ {code_set:10s} | {nom_fr:30s} | {len(cards):4d} cartes | {source_par_set[code_set]}")
    time.sleep(0.2)

# Sauvegarde brute
with open(ROOT / "data/raw/cards_index_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_cards_index, f, ensure_ascii=False, indent=2)

# Contrôle
total_recupere  = sum(len(v) for v in all_cards_index.values())
total_attendu   = df_sets["nb_cartes"].sum()
nb_tcgdex       = sum(1 for s in source_par_set.values() if s == "tcgdex")
nb_flib         = sum(1 for s in source_par_set.values() if s == "flibustier")
nb_erreurs      = sum(1 for s in source_par_set.values() if s == "ERREUR")
ecart           = total_recupere - total_attendu

print(f"\n{'='*45}")
print(f"  Sets traités   : {len(all_cards_index)}/19")
print(f"  TCGdex FR      : {nb_tcgdex} sets")
print(f"  flibustier     : {nb_flib} sets")
print(f"  Erreurs        : {nb_erreurs} sets")
print(f"{'='*45}")
print(f"  Cartes récupérées : {total_recupere}")
print(f"  Cartes attendues  : {total_attendu} (sets_master)")
print(f"  Écart             : {ecart:+d}")
if abs(ecart) < 50:
    print(f"  ✅ Écart acceptable")
else:
    print(f"  ⚠️  Écart important — vérifier les sets concernés")
print(f"{'='*45}")

  ✅ A1         | Puissance Génétique            |  286 cartes | tcgdex
  ✅ PROMO-A    | Promo-A                        |  117 cartes | flibustier
  ✅ A1a        | L’Île Fabuleuse                |   86 cartes | tcgdex
  ✅ A2         | Choc Spatio-Temporel           |  207 cartes | tcgdex
  ✅ A2a        | Lumière Triomphale             |   96 cartes | tcgdex
  ✅ A2b        | Réjouissances Rayonnantes      |  111 cartes | tcgdex
  ✅ A3         | Gardiens Astraux               |  239 cartes | tcgdex
  ✅ A3a        | Crise Interdimensionnelle      |  103 cartes | tcgdex
  ✅ A3b        | La Clairière d'Évoli           |  107 cartes | tcgdex
  ✅ A4         | Sagesse Entre Ciel et Mer      |  241 cartes | tcgdex
  ✅ A4a        | Source Secrète                 |  105 cartes | tcgdex
  ✅ A4b        | Booster de Luxe ex             |  379 cartes | flibustier
  ✅ B1         | Méga-Ascension                 |  331 cartes | flibustier
  ✅ PROMO-B    | Promo-B                        |   62 cartes | f

In [ ]:
# ============================================================
# [CELL 02] — SCRAPING CARTES POKEKALOS.FR
# Source FR principale — une page par carte
# ~3232 requêtes — prévoir ~20 minutes
# ============================================================

from bs4 import BeautifulSoup
import re

BASE_POKEKALOS = "https://www.pokekalos.fr/jeux/mobile/pocket/cartodex/extensions"

def parse_carte_pokekalos(soup, code_set, numero):
    section = soup.find("div", class_="ui grey message")
    if not section:
        return None

    divs = section.find_all("div", class_="item")

    def get_div(label):
        for div in divs:
            strong = div.find("strong")
            if strong and strong.text.strip() == label:
                return div
        return None

    def texte_apres_strong(label):
        div = get_div(label)
        if not div:
            return ""
        return " ".join(
            str(c).strip() for c in div.children
            if not (hasattr(c, 'name') and c.name == "strong")
            and str(c).strip()
        )

    def icone_src(label):
        div = get_div(label)
        if not div:
            return []
        return [img.get("src","").split("/")[-1].replace(".png","")
                for img in div.find_all("img")]

    def get_attaque_block(label_attaque):
        for i, div in enumerate(divs):
            strong = div.find("strong")
            if strong and strong.text.strip() == label_attaque:
                return divs[i+1:]
        return []

    def icones_cout(label_attaque):
        stops = {"Attaque :", "Attaque 1 :", "Attaque 2 :", "Illustration :"}
        for div in get_attaque_block(label_attaque):
            strong = div.find("strong")
            if strong and strong.text.strip() in stops and not div.find("small"):
                break
            small = div.find("small")
            if small:
                s = small.find("strong")
                if s and "Énergies" in s.text:
                    return [img.get("src","").split("/")[-1].replace(".png","")
                            for img in div.find_all("img")]
        return []

    def texte_small(label_attaque, champ):
        stops = {"Attaque :", "Attaque 1 :", "Attaque 2 :", "Illustration :"}
        for div in get_attaque_block(label_attaque):
            strong = div.find("strong")
            if strong and strong.text.strip() in stops and not div.find("small"):
                break
            small = div.find("small")
            if small:
                s = small.find("strong")
                if s and champ in s.text:
                    return " ".join(
                        str(c).strip() for c in small.children
                        if not (hasattr(c, 'name') and c.name == "strong")
                        and str(c).strip()
                    )
        return ""

    def get_talent():
        for i, div in enumerate(divs):
            strong = div.find("strong")
            if strong and strong.text.strip() == "Talent :":
                nom = " ".join(
                    str(c).strip() for c in div.children
                    if not (hasattr(c, 'name') and c.name == "strong")
                    and str(c).strip()
                ).strip()
                effet = divs[i+1].get_text(strip=True) if i+1 < len(divs) else ""
                return nom, effet
        return "", ""

    def get_effet_dresseur():
        for i, div in enumerate(divs):
            strong = div.find("strong")
            if strong and strong.text.strip() == "Carte Dresseur :":
                if i+1 < len(divs):
                    next_div = divs[i+1]
                    s = next_div.find("strong")
                    if not s or not s.text.strip():
                        return next_div.get_text(strip=True)
        return ""

    # Type carte — calculé en premier
    type_carte = "pokemon"
    div_dresseur = get_div("Carte Dresseur :")
    if div_dresseur:
        sous_type = " ".join(
            str(c).strip() for c in div_dresseur.children
            if not (hasattr(c, 'name') and c.name == "strong")
            and str(c).strip()
        ).lower()
        # Mapping sous-types non reconnus par pokekalos
        MAP_SOUS_TYPE = {
            "type de carte inconnu (5)": "stade",
        }
        sous_type = MAP_SOUS_TYPE.get(sous_type, sous_type)
        type_carte = f"dresseur-{sous_type}" if sous_type else "dresseur"

    # Effet dresseur — calculé avant tout texte_apres_strong
    effet_dresseur = get_effet_dresseur() if type_carte.startswith("dresseur") else ""

    # Talent — calculé avant tout texte_apres_strong
    talent_nom, talent_effet = get_talent()

    # Format attaque(s)
    has_a1 = get_div("Attaque 1 :") is not None
    has_a  = get_div("Attaque :")   is not None
    label_a1 = "Attaque 1 :" if has_a1 else ("Attaque :" if has_a else None)
    label_a2 = "Attaque 2 :" if has_a1 else None

    return {
        "id_carte":          f"PKT-{code_set}-{str(numero).zfill(3)}",
        "code_set":          code_set,
        "numero":            numero,
        "nom_fr":            texte_apres_strong("Nom français :"),
        "nom_en":            texte_apres_strong("Nom anglais :"),
        "type_carte":        type_carte,
        "rarete":            icone_src("Rareté :")[0] if icone_src("Rareté :") else "",
        "element":           "|".join(icone_src("Type :")) if icone_src("Type :") else "",
        "hp":                texte_apres_strong("PV :"),
        "talent_nom_fr":     talent_nom,
        "talent_effet_fr":   talent_effet,
        "attaque1_nom_fr":   texte_apres_strong(label_a1) if label_a1 else "",
        "attaque1_cout":     "|".join(icones_cout(label_a1)) if label_a1 else "",
        "attaque1_degats":   texte_small(label_a1, "Dégâts") if label_a1 else "",
        "attaque1_effet_fr": effet_dresseur if effet_dresseur else texte_small(label_a1, "Description") if label_a1 else "",
        "attaque2_nom_fr":   texte_apres_strong(label_a2) if label_a2 else "",
        "attaque2_cout":     "|".join(icones_cout(label_a2)) if label_a2 else "",
        "attaque2_degats":   texte_small(label_a2, "Dégâts") if label_a2 else "",
        "attaque2_effet_fr": texte_small(label_a2, "Description") if label_a2 else "",
        "faiblesse":         icone_src("Faiblesse :")[0] if icone_src("Faiblesse :") else "",
        "retraite":          str(len(icone_src("Retraite :"))),
        "illustrateur":      texte_apres_strong("Illustration :"),
        "source":            "pokekalos"
    }

# Scraping set par set
all_cards = []
erreurs   = []

for _, row in df_sets.iterrows():
    code_set = row["code_set"]
    slug     = row["slug_pokekalos"]
    nb       = int(row["nb_cartes"])

    if not slug or nb == 0:
        print(f"  ⏭️  {code_set} — ignoré (slug vide ou 0 cartes)")
        continue

    print(f"\n▶️  {code_set} — {row['nom_fr']} ({nb} cartes)")
    ok, ko = 0, 0

    for num in range(1, nb + 1):
        url = f"{BASE_POKEKALOS}/{slug}/cartes/{num}.html"
        try:
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "html.parser")
            carte = parse_carte_pokekalos(soup, code_set, num)
            all_cards.append(carte)
            ok += 1
        except Exception as e:
            erreurs.append({"code_set": code_set, "numero": num, "erreur": str(e)})
            ko += 1
        time.sleep(0.25)

    print(f"   ✅ {ok} | ❌ {ko}")

# Sauvegarde brute
with open(ROOT / "data/raw/cards_pokekalos_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_cards, f, ensure_ascii=False, indent=2)

print(f"\n{'='*45}")
print(f"  Total cartes scrapées : {len(all_cards)}")
print(f"  Erreurs               : {len(erreurs)}")
print(f"{'='*45}")


▶️  A1 — Puissance Génétique (286 cartes)
   ✅ 286 | ❌ 0

▶️  PROMO-A — Promo-A (117 cartes)
   ✅ 117 | ❌ 0

▶️  A1a — L’Île Fabuleuse (86 cartes)
   ✅ 86 | ❌ 0

▶️  A2 — Choc Spatio-Temporel (207 cartes)
   ✅ 207 | ❌ 0

▶️  A2a — Lumière Triomphale (96 cartes)
   ✅ 96 | ❌ 0

▶️  A2b — Réjouissances Rayonnantes (111 cartes)
   ✅ 111 | ❌ 0

▶️  A3 — Gardiens Astraux (239 cartes)
   ✅ 239 | ❌ 0

▶️  A3a — Crise Interdimensionnelle (103 cartes)
   ✅ 103 | ❌ 0

▶️  A3b — La Clairière d'Évoli (107 cartes)
   ✅ 107 | ❌ 0

▶️  A4 — Sagesse Entre Ciel et Mer (241 cartes)
   ✅ 241 | ❌ 0

▶️  A4a — Source Secrète (105 cartes)
   ✅ 105 | ❌ 0

▶️  A4b — Booster de Luxe ex (379 cartes)
   ✅ 379 | ❌ 0

▶️  B1 — Méga-Ascension (331 cartes)
   ✅ 331 | ❌ 0

▶️  PROMO-B — Promo-B (62 cartes)
   ✅ 62 | ❌ 0

▶️  B1a — Embrasement Écarlate (103 cartes)
   ✅ 103 | ❌ 0

▶️  B2 — Parade Onirique (234 cartes)
   ✅ 234 | ❌ 0

▶️  B2a — Merveilles de Paldea (131 cartes)
   ✅ 131 | ❌ 0

▶️  B2b — Méga-Rayonnemen

In [90]:
# ============================================================
# [CELL 03] — FUSION + EXPORT cards_pocket_fr.csv
# Source principale : cards_pokekalos_raw.json
# Complément : cards_index_raw.json (TCGdex + flibustier)
# ============================================================

import json
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()

# Chargement JSON bruts
with open(ROOT / "data/raw/cards_pokekalos_raw.json", encoding="utf-8") as f:
    cards_pokekalos = json.load(f)

with open(ROOT / "data/raw/cards_index_raw.json", encoding="utf-8") as f:
    cards_index = json.load(f)  # {code_set: [cards...]}

# Index TCGdex/flibustier par (code_set, numero) pour enrichissement
index_complement = {}
for code_set, cards in cards_index.items():
    for c in cards:
        num = c.get("localId") or c.get("id", "")
        try:
            num = int(num)
        except:
            continue
        index_complement[(code_set, num)] = c

# Construction DataFrame principal depuis pokekalos
df = pd.DataFrame(cards_pokekalos)

# Enrichissement image_url depuis TCGdex/flibustier
def enrich(row):
    key = (row["code_set"], int(row["numero"]))
    comp = index_complement.get(key, {})
    row["image_url"] = comp.get("image", comp.get("imageUrl", ""))
    return row

df = df.apply(enrich, axis=1)

# Nettoyage + typage
df["numero"] = pd.to_numeric(df["numero"], errors="coerce").astype("Int64")
df["hp"]     = pd.to_numeric(df["hp"], errors="coerce").astype("Int64")

# Colonnes finales ordonnées — attaque2 + illustrateur ajoutés
COLONNES = [
    "id_carte", "code_set", "numero", "nom_fr", "nom_en", "type_carte",
    "rarete", "element", "hp",
    "talent_nom_fr", "talent_effet_fr",
    "attaque1_nom_fr", "attaque1_cout", "attaque1_degats", "attaque1_effet_fr",
    "attaque2_nom_fr", "attaque2_cout", "attaque2_degats", "attaque2_effet_fr",
    "faiblesse", "retraite",
    "illustrateur", "image_url", "source"
]

# Ajouter colonnes manquantes si absentes
for col in COLONNES:
    if col not in df.columns:
        df[col] = ""

df = df[COLONNES]
df["type_carte"] = df["type_carte"].replace(
    "dresseur-type de carte inconnu (5)", "dresseur-stade"
)

# Export
out = ROOT / "data/processed/cards_pocket_fr.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")

# Contrôle
print(f"✅ Export : {out}")
print(f"📊 Lignes              : {len(df)}")
print(f"📊 Colonnes            : {len(df.columns)}")
print(f"❌ nom_fr vides        : {df['nom_fr'].eq('').sum()}")
print(f"❌ numero vides        : {df['numero'].isna().sum()}")
print(f"❌ code_set vides      : {df['code_set'].eq('').sum()}")
print(f"❌ attaque1 vides      : {df['attaque1_nom_fr'].eq('').sum()}")
print(f"❌ illustrateur vides  : {df['illustrateur'].eq('').sum()}")
doublons = df.duplicated(subset=["code_set","numero"]).sum()
print(f"❌ Doublons clé        : {doublons}")
print(f"\n📊 Types cartes :")
print(df["type_carte"].value_counts().to_string())
print(f"\n📊 Raretés :")
print(df["rarete"].value_counts().to_string())
print(f"\n📋 Aperçu :")
display(df.head(5))

✅ Export : i:\Drive partagés\Prismora - Drive Principal\01_Clients\pokemon-bdd\data\processed\cards_pocket_fr.csv
📊 Lignes              : 3289
📊 Colonnes            : 24
❌ nom_fr vides        : 0
❌ numero vides        : 0
❌ code_set vides      : 0
❌ attaque1 vides      : 451
❌ illustrateur vides  : 4
❌ Doublons clé        : 0

📊 Types cartes :
type_carte
pokemon                     3021
dresseur-supporter           175
dresseur-objet                43
dresseur-outil pokémon        30
dresseur-objet (fossile)      12
dresseur-stade                 8

📊 Raretés :
rarete
diamant     2391
etoile       598
shiny        271
couronne      29

📋 Aperçu :


,id_carte,code_set,numero,nom_fr,nom_en,type_carte,rarete,element,hp,talent_nom_fr,...,attaque1_effet_fr,attaque2_nom_fr,attaque2_cout,attaque2_degats,attaque2_effet_fr,faiblesse,retraite,illustrateur,image_url,source
0,PKT-A1-001,A1,1,Bulbizarre,Bulbasaur,pokemon,diamant,plante,70,,...,,,,,,feu,1,Narumi Sato,https://assets.tcgdex.net/fr/tcgp/A1/001,pokekalos
1,PKT-A1-002,A1,2,Herbizarre,Ivysaur,pokemon,diamant,plante,90,,...,,,,,,feu,2,Kurata So,https://assets.tcgdex.net/fr/tcgp/A1/002,pokekalos
2,PKT-A1-003,A1,3,Florizarre,Venusaur,pokemon,diamant,plante,160,,...,Soignez 30 dégâts de ce Pokémon.,,,,,feu,3,Ryota Murayama,https://assets.tcgdex.net/fr/tcgp/A1/003,pokekalos
3,PKT-A1-004,A1,4,Florizarre-ex,Venusaur ex,pokemon,diamant,plante,190,,...,,Pousse Géante,plante|plante|incolore|incolore,100,Soignez 30 dégâts de ce Pokémon.,feu,3,PLANETA CG Works,https://assets.tcgdex.net/fr/tcgp/A1/004,pokekalos
4,PKT-A1-005,A1,5,Chenipan,Caterpie,pokemon,diamant,plante,50,,...,Ajoutez au hasard un Pokémon Plante de votre d...,,,,,feu,1,Miki Tanaka,https://assets.tcgdex.net/fr/tcgp/A1/005,pokekalos


In [87]:
print(df["type_carte"].value_counts().to_string())

type_carte
pokemon                               3021
dresseur-supporter                     175
dresseur-objet                          43
dresseur-outil pokémon                  30
dresseur-objet (fossile)                12
dresseur-type de carte inconnu (5)       8


In [86]:
url = "https://www.pokekalos.fr/jeux/mobile/pocket/cartodex/extensions/parade-onirique/cartes/153.html"
resp = requests.get(url, timeout=15)
soup = BeautifulSoup(resp.text, "html.parser")
section = soup.find("div", class_="ui grey message")
for div in section.find_all("div", class_="item"):
    strong = div.find("strong")
    if strong and "Dresseur" in strong.text:
        print(repr(div.get_text(strip=True)))

'Carte Dresseur :Type de carte inconnu (5)'
